# Herramientas · Criptografía

Firma digital y cifrado asimétrico con la librería `cryptography` (ejercicios de [El Libro De Python](https://ellibrodepython.com/ejemplos-ejercicios-python)).

## 1. Firma digital con otra clave pública

Firmar un mensaje con una clave privada de curva elíptica y verificarlo con su clave pública. Después, intentar verificar la firma con una clave pública distinta y explicar qué sucede.

La verificación falla (`InvalidSignature`): la firma solo la puede validar la clave pública emparejada con la privada que firmó, así que otra clave no puede dar por buena una firma que no hizo su pareja.

In [1]:
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec

privada = ec.generate_private_key(ec.SECP256R1())
publica = privada.public_key()
otra_publica = ec.generate_private_key(ec.SECP256R1()).public_key()
mensaje = b"Envia 1 Euro al IBAN 123"
firma = privada.sign(mensaje, ec.ECDSA(hashes.SHA256()))

def verifica(clave, firma, mensaje):
    try:
        clave.verify(firma, mensaje, ec.ECDSA(hashes.SHA256()))
        return "Verificación OK"
    except InvalidSignature:
        return "Verificación NOK"

print(verifica(publica, firma, mensaje))
print(verifica(publica, firma, b"Envia 100 Euros al IBAN 456"))
print(verifica(otra_publica, firma, mensaje))

Verificación OK
Verificación NOK
Verificación NOK


## 2. Descifrar con otra clave privada

Cifrar un mensaje con la clave pública RSA del destinatario y descifrarlo con su clave privada. Después, intentar descifrarlo con otra clave privada y explicar qué sucede.

El descifrado falla (`ValueError`): solo la clave privada emparejada con la pública usada para cifrar puede recuperar el mensaje. El relleno OAEP detecta que el resultado no es válido en lugar de devolver datos basura.

In [2]:
from cryptography.hazmat.primitives.asymmetric import padding, rsa

privada = rsa.generate_private_key(public_exponent=65537, key_size=2048)
publica = privada.public_key()
otra_privada = rsa.generate_private_key(public_exponent=65537, key_size=2048)
relleno = padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)

cifrado = publica.encrypt(b"Mensaje secreto de El Libro de Python", relleno)
print(privada.decrypt(cifrado, relleno))
try:
    otra_privada.decrypt(cifrado, relleno)
except ValueError as e:
    print("ValueError:", e)

b'Mensaje secreto de El Libro de Python'
ValueError: Decryption failed


## 3. Cifrado César y su ruptura por frecuencias

El cifrado César sustituye cada letra por la que está $k$ posiciones más adelante en el alfabeto: $c_i = (m_i + k) \bmod 26$. Escribir funciones para normalizar un texto (mayúsculas sin tildes ni signos), cifrarlo y descifrarlo. Romperlo sin conocer la clave probando los 26 desplazamientos y quedándose con el que da unas frecuencias de letras más parecidas a las del castellano según el estadístico $\chi^2 = \sum_{l} \frac{(O_l - E_l)^2}{E_l}$.

In [3]:
import string
import unicodedata
from collections import Counter

ABC = string.ascii_uppercase
# frecuencias aproximadas (%) de las letras en castellano, con la ñ contada como n
FRECUENCIAS = dict(zip(ABC, [12.53, 1.42, 4.68, 5.86, 13.68, 0.69, 1.01, 0.70, 6.25, 0.44, 0.02, 4.97, 3.15,
                             7.02, 8.68, 2.51, 0.88, 6.87, 7.98, 4.63, 3.93, 0.90, 0.01, 0.22, 0.90, 0.52]))
TEXTO = (
    "La criptografía es el arte de escribir mensajes que solo pueda leer quien conoce un secreto. Durante siglos fue "
    "una herramienta de reyes, generales y diplomáticos, que confiaban sus órdenes a mensajeros que podían ser "
    "capturados por el enemigo. Por eso el contenido debía resultar ilegible para cualquiera que no tuviera la clave. "
    "Los primeros sistemas sustituían cada letra por otra siguiendo una regla sencilla, como desplazar el alfabeto unas "
    "cuantas posiciones. Su debilidad es que conservan la frecuencia de las letras: en un texto en castellano la letra "
    "más repetida suele ser la e, seguida de la a y de la o, y esa huella sobrevive al cifrado. Un analista paciente "
    "solo tiene que contar las letras del mensaje interceptado y compararlas con las del idioma para descubrir la clave. "
    "Los cifrados polialfabéticos intentaron ocultar esa huella usando varios alfabetos que se alternan según una "
    "palabra clave, de modo que una misma letra se cifra de formas distintas en cada posición. Durante mucho tiempo se "
    "consideraron indescifrables, hasta que se comprendió que, si se conoce la longitud de la clave, el mensaje se "
    "puede separar en columnas y cada columna vuelve a ser un cifrado sencillo que cede ante el recuento de frecuencias."
)

def normaliza(texto):
    sin_tildes = unicodedata.normalize("NFD", texto.upper())
    return "".join(c for c in sin_tildes if c in ABC)

def cesar(texto, clave):
    return "".join(ABC[(ABC.index(c) + clave) % 26] for c in normaliza(texto))

def chi2(texto):
    cuenta, n = Counter(texto), len(texto)
    return sum((cuenta[l] - n * f / 100) ** 2 / (n * f / 100) for l, f in FRECUENCIAS.items())

def rompe_cesar(cifrado):
    return min(range(26), key=lambda k: chi2(cesar(cifrado, -k)))

cifrado = cesar(TEXTO, 3)
print(normaliza(TEXTO)[:60], cifrado[:60], cesar(cifrado, -3)[:60], sep="\n")
print(Counter(cifrado).most_common(4), len(cifrado))
assert cesar(cifrado, -3) == normaliza(TEXTO)
assert all(rompe_cesar(cesar(TEXTO, k)) == k for k in range(26))
assert all(rompe_cesar(cesar(TEXTO[:200], k)) == k for k in range(26))
print("clave encontrada:", rompe_cesar(cifrado))

LACRIPTOGRAFIAESELARTEDEESCRIBIRMENSAJESQUESOLOPUEDALEERQUIE
ODFULSWRJUDILDHVHODUWHGHHVFULELUPHQVDMHVTXHVRORSXHGDOHHUTXLH
LACRIPTOGRAFIAESELARTEDEESCRIBIRMENSAJESQUESOLOPUEDALEERQUIE
[('H', 148), ('D', 128), ('V', 84), ('R', 72)] 1015
clave encontrada: 3


## 4. Cifrado de Vigenère y su ruptura

Vigenère usa una palabra clave: la letra $i$ del mensaje se desplaza según la letra $i \bmod L$ de la clave, de modo que una misma letra se cifra de formas distintas. Escribir las funciones de cifrado y descifrado y romperlo sin la clave en dos pasos: estimar su longitud $L$ con el índice de coincidencia (la probabilidad de que dos letras al azar de un texto coincidan, unos 0,07 en castellano y 0,038 en un texto aleatorio), separando el cifrado en $L$ columnas, y descifrar cada columna, que es un César, con la función del ejercicio anterior.

In [4]:
def vigenere(texto, clave, signo=1):
    claves = [ABC.index(c) for c in normaliza(clave)]
    return "".join(ABC[(ABC.index(c) + signo * claves[i % len(claves)]) % 26] for i, c in enumerate(normaliza(texto)))

def indice_coincidencia(texto):
    n = len(texto)
    return sum(v * (v - 1) for v in Counter(texto).values()) / (n * (n - 1))

def longitud_clave(cifrado, maximo=12, umbral=0.06):
    for L in range(1, maximo + 1):
        if sum(indice_coincidencia(cifrado[i::L]) for i in range(L)) / L > umbral:
            return L

def rompe_vigenere(cifrado):
    L = longitud_clave(cifrado)
    return "".join(ABC[rompe_cesar(cifrado[i::L])] for i in range(L))

cifrado = vigenere(TEXTO, "SECRETO")
print(cifrado[:60], vigenere(cifrado, "SECRETO", -1)[:60], sep="\n")
print(Counter(cifrado).most_common(4))
print([round(sum(indice_coincidencia(cifrado[i::L]) for i in range(L)) / L, 3) for L in range(1, 10)])
print(indice_coincidencia(normaliza(TEXTO)), longitud_clave(cifrado), rompe_vigenere(cifrado))
assert vigenere(cifrado, "SECRETO", -1) == normaliza(TEXTO)
assert all(rompe_vigenere(vigenere(TEXTO, clave)) == clave for clave in ["SECRETO", "CLAVE", "PYTHON", "LIMON"])

DEEIMIHGKTRJBOWWGCEKHWHGVWVFAFKIQXBKELVWJIWWQCSIIWHCCIXFIYKV
LACRIPTOGRAFIAESELARTEDEESCRIBIRMENSAJESQUESOLOPUEDALEERQUIE
[('E', 77), ('I', 70), ('G', 70), ('W', 69)]
[0.046, 0.046, 0.046, 0.046, 0.046, 0.046, 0.075, 0.046, 0.046]
0.07634204875584186 7 SECRETO


## 5. Resúmenes SHA-256 e integridad de ficheros

Calcular con `hashlib` el resumen SHA-256 de varios mensajes y comprobar el valor de referencia de `b"abc"`. Comprobar el efecto avalancha (cambiar un carácter cambia más o menos la mitad de los 256 bits). Escribir una función que calcule el SHA-256 de un fichero leyéndolo por bloques, compararla con `hashlib.file_digest` y usarla para detectar que se ha modificado un solo bit del fichero.

In [5]:
import hashlib
from pathlib import Path

print(hashlib.sha256(b"hola").hexdigest())
assert hashlib.sha256(b"abc").hexdigest() == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"

a, b = hashlib.sha256(b"Transferir 100 euros").digest(), hashlib.sha256(b"Transferir 900 euros").digest()
print(a.hex()[:32], b.hex()[:32], sum(bin(x ^ y).count("1") for x, y in zip(a, b)), "de 256 bits distintos")

def sha256_fichero(ruta, bloque=65536):
    resumen = hashlib.sha256()
    with open(ruta, "rb") as f:
        while trozo := f.read(bloque):
            resumen.update(trozo)
    return resumen.hexdigest()

ruta = Path("datos.bin")
ruta.write_bytes(bytes(range(256)) * 4000)
original = sha256_fichero(ruta)
with open(ruta, "rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == original
datos = bytearray(ruta.read_bytes())
datos[500_000] ^= 1
ruta.write_bytes(datos)
print(ruta.stat().st_size, original[:16], sha256_fichero(ruta)[:16], sha256_fichero(ruta) == original)
ruta.unlink()

b221d9dbb083a7f33428d7c2a3c3198ae925614d70210e28716ccaa7cd4ddb79
c4748d2c695439b1ce9e309a1d2c400d 424d50780ffb05a9fe256165dfd976bc 132 de 256 bits distintos
1024000 062af9ccd890ba3d 0526fc86c9b8994b False


## 6. Autenticar mensajes con HMAC

Un resumen protege frente a cambios accidentales, pero quien modifica un mensaje puede recalcular su SHA-256. Firmar un mensaje con `hmac` y una clave secreta compartida, y verificarlo con `hmac.compare_digest`, cuyo tiempo no depende de la posición del primer carácter distinto y así no da pistas a quien mide el tiempo de respuesta. Comprobar que un mensaje alterado o un SHA-256 recalculado por un atacante no pasan la verificación, y validar la implementación con el caso de prueba 2 del RFC 4231.

In [6]:
import hmac
import secrets

def firma(clave, mensaje):
    return hmac.new(clave, mensaje, hashlib.sha256).hexdigest()

def verifica_hmac(clave, mensaje, etiqueta):
    return hmac.compare_digest(firma(clave, mensaje), etiqueta)

clave = secrets.token_bytes(32)
mensaje, alterado = b"pedido=42&importe=19.99", b"pedido=42&importe=0.01"
etiqueta = firma(clave, mensaje)
print(len(etiqueta), verifica_hmac(clave, mensaje, etiqueta), verifica_hmac(clave, alterado, etiqueta))
print(verifica_hmac(clave, alterado, hashlib.sha256(alterado).hexdigest()), verifica_hmac(secrets.token_bytes(32), mensaje, etiqueta))
assert firma(b"Jefe", b"what do ya want for nothing?") == "5bdcc146bf60754e6a042426089575c75a003f089d2739839dec58b964ec3843"

64 True False
False False


## 7. Contraseñas: generarlas con `secrets` y guardarlas con scrypt

Generar contraseñas aleatorias seguras con `secrets` (nunca con `random`, cuya secuencia es predecible) que tengan al menos una minúscula, una mayúscula, un dígito y un símbolo, y un token para un enlace de recuperación. Para guardarlas, calcular `hashlib.scrypt` con una sal aleatoria distinta para cada usuario, de modo que dos usuarios con la misma contraseña tengan resúmenes distintos, y comprobar una contraseña con `hmac.compare_digest`. Comparar el tiempo de scrypt, lento a propósito para frenar los ataques de fuerza bruta, con el de SHA-256.

In [7]:
import timeit

SIMBOLOS = "!@#$%&*-_"

def genera_contraseña(longitud=16):
    alfabeto = string.ascii_letters + string.digits + SIMBOLOS
    while True:
        contraseña = "".join(secrets.choice(alfabeto) for _ in range(longitud))
        if all(any(c in grupo for c in contraseña) for grupo in (string.ascii_lowercase, string.ascii_uppercase, string.digits, SIMBOLOS)):
            return contraseña

SCRYPT = {"n": 2**14, "r": 8, "p": 1, "dklen": 32}

def protege(contraseña):
    sal = secrets.token_bytes(16)
    return sal + hashlib.scrypt(contraseña.encode(), salt=sal, **SCRYPT)

def comprueba(contraseña, guardado):
    sal, resumen = guardado[:16], guardado[16:]
    return hmac.compare_digest(hashlib.scrypt(contraseña.encode(), salt=sal, **SCRYPT), resumen)

nuevas = [genera_contraseña() for _ in range(100)]
assert len(set(nuevas)) == 100 and all(len(c) == 16 for c in nuevas)
print(len(secrets.token_urlsafe(32)), "caracteres en el token del enlace de recuperación")

ana, luis = protege("caballo correcto"), protege("caballo correcto")
print(len(ana), ana != luis, comprueba("caballo correcto", ana), comprueba("caballo incorrecto", ana))
t_scrypt = min(timeit.repeat(lambda: protege("caballo correcto"), repeat=3, number=1))
t_sha = min(timeit.repeat(lambda: hashlib.sha256(b"caballo correcto").digest(), repeat=3, number=10_000)) / 10_000
print(f"scrypt {t_scrypt * 1000:.0f} ms, SHA-256 {t_sha * 1e6:.2f} µs: {t_scrypt / t_sha:.0f} veces más lento")

43 caracteres en el token del enlace de recuperación


48 True True False


scrypt 41 ms, SHA-256 0.45 µs: 89425 veces más lento


## 8. Cifrado simétrico con Fernet

Fernet, de la librería `cryptography`, cifra con AES y añade un HMAC y la fecha de creación al token. Generar una clave, cifrar y descifrar un mensaje, y comprobar que un token manipulado, una clave equivocada o un token caducado (opción `ttl`) lanzan `InvalidToken`. Derivar después la clave a partir de una contraseña y una sal con PBKDF2 para no tener que guardar la clave.

In [8]:
import base64
import time

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

clave = Fernet.generate_key()
fernet = Fernet(clave)
token = fernet.encrypt("Reunión a las 18:00 en la biblioteca".encode())
print(len(clave), len(token), fernet.decrypt(token).decode())

bytes_token = bytearray(base64.urlsafe_b64decode(token))
bytes_token[30] ^= 1
manipulado = base64.urlsafe_b64encode(bytes_token)
viejo = fernet.encrypt_at_time(b"oferta de ayer", int(time.time()) - 86400)
for nombre, f, t, ttl in [("manipulado", fernet, manipulado, None), ("otra clave", Fernet(Fernet.generate_key()), token, None),
                          ("caducado", fernet, viejo, 3600)]:
    try:
        f.decrypt(t, ttl=ttl)
    except InvalidToken:
        print(nombre, "-> InvalidToken")

def clave_de_contraseña(contraseña, sal):
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=sal, iterations=600_000)
    return base64.urlsafe_b64encode(kdf.derive(contraseña.encode()))

sal = secrets.token_bytes(16)
token = Fernet(clave_de_contraseña("caballo correcto", sal)).encrypt(b"Mis notas privadas")
print(Fernet(clave_de_contraseña("caballo correcto", sal)).decrypt(token))
assert fernet.decrypt(viejo) == b"oferta de ayer"

44 140 Reunión a las 18:00 en la biblioteca
manipulado -> InvalidToken
otra clave -> InvalidToken
caducado -> InvalidToken


b'Mis notas privadas'
